# 03 · detsim

**Purpose.** Raw waveforms after detsim, fibers and PMTs on the same nexus events.

**Inputs.** `<sample>_<i>.detsim.h5` (new: `$ICA_OUT/...`; 2025: `$OLD_IC_DIR/detsim`, PMTs in `detsim/PMTs`).

**Outputs.** None.

**What is checked.** stored detsim config of every file; S1 and S2 integrals per event for fibers and PMTs on the same events; the S1 placeholder (60 PMT waveforms copied into 108 fiber channels).

**Next step.** `04_hypathia`. Run with `./submit.sh detsim <fibers|pmts> <sample>`.

In [ ]:
# Setup: run `source env.sh` before starting Jupyter, or pick the IC-3.8-2024-06-08 kernel.
import os, sys, glob
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from icana import paths, checks, fits
print("ICDIR =", paths.ICDIR or "NOT SET (source env.sh)")

In [ ]:
EL_GAIN = paths.EL_GAIN   # gain of the new runs to read (folder <det>_g<EL_GAIN>)
SAMPLE, I = "20250624_Next100_1Kev_ELoff_Kr", 1     # file index; the 1Kev sample exists for both detectors
USE_NEW = False                                       # True: files made by this repository
def detsim_file(det):
    fs = paths.new_files(SAMPLE, det, "detsim", EL_GAIN) if USE_NEW else paths.old_files(SAMPLE, det, "detsim")
    return [f for f in fs if paths.file_index(f.replace("_PMTs", "")) == I][0]
f_fib, f_pmt = detsim_file("fibers"), detsim_file("pmts"); print(f_fib); print(f_pmt)
checks.summary_ic([f_fib, f_pmt])

In [ ]:
import tables as tb
def integrals(path, node, n_events=200, s1=(990, 1015)):
    # EP waveforms: 100 ns bins, S1 at 100 us (bin 1000); S2 after it
    with tb.open_file(path) as h:
        wf = getattr(h.root, node)
        out = []
        for e in range(min(n_events, wf.shape[0])):
            w = wf[e].astype(float).sum(axis=0)
            out.append((w[s1[0]:s1[1]].sum(), w[s1[1]:].sum()))
    return np.array(out)
fib = integrals(f_fib, "fibrd"); pmt = integrals(f_pmt, "pmtrd")
print(f"median S1: fibers {np.median(fib[:,0]):.1f}, PMTs {np.median(pmt[:,0]):.1f} adc, ratio {np.median(fib[:,0])/np.median(pmt[:,0]):.2f}")
print(f"median S2: fibers {np.median(fib[:,1]):.0f}, PMTs {np.median(pmt[:,1]):.0f} adc, ratio {np.median(fib[:,1])/np.median(pmt[:,1]):.2f}")

### The fiber S1 is a placeholder
`detsim.py:scale_and_repeat_s1` builds the 108 fiber S1 waveforms from the 60 PMT S1 waveforms: `scaling_factor = 1` (the 60/108 line is commented out), all 60 copied, then the first 48 again.
So fiber S1 / PMT S1 ≈ 1 + (light in PMTs 0-47)/(all PMT light), about 1.8, by construction. The S2 ratio comes from the fiber S2 table and is physics.
Check below: fiber channels 60-107 must be identical to channels 0-47 in the S1 window.

In [ ]:
with tb.open_file(f_fib) as h:
    w = h.root.fibrd[0].astype(float)
s1 = w[:, 990:1015]
print("channels 60-107 equal to 0-47 in the S1 window:", np.allclose(s1[60:108], s1[0:48]))